# DeepEval full-pipeline case analysis

This notebook inspects one `full_pipeline_eval.py` run. It separates pipeline/API errors from metric failures, lists every score below its recorded threshold, preserves the judge's reason, and supports detailed inspection of any case.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

RESULTS_ROOT = Path("eval/deepEval/full_pipeline_results")
RUN_ID = "2026-10-24_13-42-39_huggingface_Qwen_Qwen2.5-7B-Instruct_featherless-ai"
RUN_DIR = RESULTS_ROOT / RUN_ID
CASES_PATH = RUN_DIR / "cases.jsonl"
SUMMARY_PATH = RUN_DIR / "summary.json"
MANIFEST_PATH = RUN_DIR / "manifest.json"


def load_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


records = load_jsonl(CASES_PATH)
summary = load_json(SUMMARY_PATH)
manifest = load_json(MANIFEST_PATH)

print(f"Run: {manifest['run_id']}")
print(f"Cases: {len(records)}")
print(f"Generator: {manifest['generation_config']['provider']}/{manifest['generation_config']['model']}")
print(f"Judge: {manifest['judge_model']}")
print(f"Context-k: {manifest['generation_config']['context_k']}")
print(f"Prompt version: {manifest['generation_config']['prompt_version']}")
print(f"Default threshold: {manifest['threshold']}")

In [ ]:
metric_summary = pd.DataFrame(
    [
        {
            "metric": name,
            "mean_score": values["mean_score"],
            "pass_rate": values["pass_rate"],
            "scored_cases": values["scored_cases"],
            "passed_cases": round(values["pass_rate"] * values["scored_cases"]),
        }
        for name, values in summary["metrics"].items()
    ]
)

run_summary = pd.DataFrame(
    {
        "value": {
            "attempted_cases": summary["cases"],
            "successful_pipeline_cases": summary["successful_pipeline_cases"],
            "pipeline_error_cases": summary["pipeline_error_cases"],
            "triad_mean_score": summary["triad_mean_score"],
            "prompt_tokens": summary["prompt_tokens"],
            "completion_tokens": summary["completion_tokens"],
            "pipeline_latency_seconds": summary["pipeline_latency_seconds"],
        }
    }
)

display(run_summary)
display(metric_summary.style.format({"mean_score": "{:.3f}", "pass_rate": "{:.1%}"}))

In [ ]:
case_rows = []
metric_rows = []

for record in records:
    case = record["case"]
    case_id = case.get("id")
    case_rows.append(
        {
            "case_id": case_id,
            "status": record["status"],
            "doc_id": case.get("doc_id"),
            "category": case.get("category"),
            "query": case.get("query"),
            "error": record.get("error"),
        }
    )
    for metric_name, result in record.get("metrics", {}).items():
        metric_rows.append(
            {
                "case_id": case_id,
                "metric": metric_name,
                "score": result.get("score"),
                "threshold": result.get("threshold"),
                "passed": result.get("passed"),
                "judge_reason": result.get("reason"),
                "metric_error": result.get("error"),
                "evaluation_model": result.get("evaluation_model"),
            }
        )

cases_df = pd.DataFrame(case_rows)
metrics_df = pd.DataFrame(metric_rows)
error_cases_df = cases_df[cases_df["status"] != "success"].copy()
failed_metrics_df = metrics_df[
    (metrics_df["score"].isna())
    | (metrics_df["score"] < metrics_df["threshold"])
    | (~metrics_df["passed"].fillna(False))
].sort_values(["case_id", "metric"])

print(f"Pipeline/API error cases: {len(error_cases_df)}")
display(error_cases_df)
print(f"Below-threshold or errored metric results: {len(failed_metrics_df)}")
display(failed_metrics_df.style.format({"score": "{:.3f}", "threshold": "{:.3f}"}))

## Detailed failure inspection

The next view prints every failed metric together with the question, retrieved context, actual answer, score, threshold, and the judge's complete reason.

In [ ]:
records_by_id = {
    record["case"].get("id"): record for record in records
}

for row in failed_metrics_df.to_dict("records"):
    record = records_by_id[row["case_id"]]
    case = record["case"]
    retrieval_context = record.get("retrieval_context", [])
    context_preview = "\n\n".join(retrieval_context[:2]) if retrieval_context else "None"
    if len(retrieval_context) > 2:
        context_preview += f"\n\n... ({len(retrieval_context) - 2} more chunks)"
    
    display(
        Markdown(
            f"""### {row['case_id']} — {row['metric']}

**Score:** `{row['score']}` &nbsp; **Threshold:** `{row['threshold']}` &nbsp; **Passed:** `{row['passed']}`

**Question**

{case.get('query')}

**Retrieved context (first 2 chunks)**

```text
{context_preview}
```

**Actual answer**

{record.get('actual_output')}

**Judge reason**

{row['judge_reason'] or row['metric_error'] or 'No reason recorded.'}

---
"""
        )
    )

## Inspect any individual case

Change `CASE_ID` to inspect all recorded metrics and reasons for one case, including cases that passed.

In [ ]:
def inspect_case(case_id):
    record = records_by_id[case_id]
    case = record["case"]
    display(Markdown(f"## {case_id}: {case['query']}"))
    display(
        pd.DataFrame(
            {
                "value": {
                    "status": record["status"],
                    "document": case.get("doc_id"),
                    "category": case.get("category"),
                    "actual_output": record.get("actual_output"),
                    "pipeline_error": record.get("error"),
                }
            }
        )
    )
    case_metrics = metrics_df[metrics_df["case_id"] == case_id]
    if case_metrics.empty:
        print("No metric results were recorded for this case.")
    else:
        display(case_metrics.style.format({"score": "{:.3f}", "threshold": "{:.3f}"}))
    
    display(Markdown("### Retrieved context"))
    retrieval_context = record.get("retrieval_context", [])
    for index, context in enumerate(retrieval_context, start=1):
        display(Markdown(f"**Context {index}**\n\n```text\n{context}\n```"))
    return record


CASE_ID = "q007"
selected_record = inspect_case(CASE_ID)

## All case scores

This matrix provides a quick comparison across all successfully scored cases. Missing values correspond to pipeline/API errors or metric errors.

In [ ]:
score_matrix = metrics_df.pivot(index="case_id", columns="metric", values="score")
score_matrix = cases_df[["case_id", "status", "doc_id", "category"]].merge(
    score_matrix, left_on="case_id", right_index=True, how="left"
)
metric_columns = [column for column in manifest["metrics"] if column in score_matrix.columns]
display(
    score_matrix.style
    .format({column: "{:.3f}" for column in metric_columns}, na_rep="—")
)